# Eksperimen RQ1 NER — Colab

RQ1: apakah normalisasi dan augmentasi teks informal memperbaiki ekstraksi entitas?
Varian dan seed diambil dari `configs/experiments/rq1_ner.yaml` (10 varian × 3 seed).
Tiap varian dinilai pada NERGrit test (formal) dan `nergrit_test_informal` (informal sintetis).

Cara pakai sama dengan `train_ner_colab.ipynb`: runtime GPU, kode sudah di-push, jalankan semua sel.

In [ ]:
# Pengaturan
REPO_URL = "https://github.com/danenftyessir/indonesian-hoax-checker.git"
BRANCH = "feat/ner"

# Folder Drive tim: tubes_kelompok9 > hasil_training
DRIVE_FOLDER_ID = "1Eq9C3SIW42xYpRn5UVAwoiLrRkOY52Pn"

## 1. Cek GPU dan login Google (untuk upload hasil)

In [ ]:
!nvidia-smi

from google.colab import auth
auth.authenticate_user()

## 2. Ambil kode dan install dependensi

Kalau repo private, simpan token GitHub di Colab Secrets dengan nama `GITHUB_TOKEN`.

In [ ]:
import os

# Pakai token kalau repo private
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    url = REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
%cd /content/repo
!git pull -q
!git log --oneline -1
!pip install -q -e ".[training]"

## 3. Siapkan data NERGrit dan NERP

In [ ]:
!python -m indonesian_hoax_checker.ner.prepare

## 4. Jalankan semua varian

Varian yang sudah punya ringkasan dilewati, jadi sel ini bisa dijalankan ulang bila sesi Colab terputus.

In [ ]:
import yaml

plan = yaml.safe_load(open("configs/experiments/rq1_ner.yaml"))
seeds = " ".join(str(s) for s in plan["seeds"])
for config in plan["variants"]:
    name = yaml.safe_load(open(config))["name"]
    if os.path.exists(f"outputs/metrics/{name}_summary.json"):
        print("lewati", name)
        continue
    !python -m indonesian_hoax_checker.ner.train --config {config} --seeds {seeds} 2>&1 | grep -E "best epoch|Error|error"
    !rm -rf artifacts/checkpoints/ner/{name}_seed*

## 5. Ringkasan RQ1 (F1, rata-rata ± simpangan baku)

In [ ]:
import json
import pandas as pd

rows = []
for config in plan["variants"]:
    name = yaml.safe_load(open(config))["name"]
    path = f"outputs/metrics/{name}_summary.json"
    if not os.path.exists(path):
        continue
    s = json.load(open(path))
    row = {"varian": name, "formal": f"{s['test_f1']['mean']:.4f} ± {s['test_f1']['std']:.4f}"}
    for key, v in s.get("extra_f1", {}).items():
        row[key] = f"{v['mean']:.4f} ± {v['std']:.4f}"
    rows.append(row)
table = pd.DataFrame(rows)
table.to_csv("/content/rq1_summary.csv", index=False)
table

In [ ]:
# Error analysis seed 42 tiap varian (disimpan ke outputs/metrics/*_errors.json)
!python -m indonesian_hoax_checker.ner.error_analysis outputs/predictions/rq1_*_seed42.conll --examples 0 > /dev/null
!ls outputs/metrics | grep errors

## 6. Upload hasil ke Drive tim

Isi `hasil_training/rq1_ner_<waktu>/`: `rq1_summary.csv` dan `rq1_ner_results.zip` (metrik, prediksi, error analysis semua run). Checkpoint tidak ikut supaya ukurannya kecil.

In [ ]:
from datetime import datetime

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

drive = build("drive", "v3")


def make_folder(name, parent_id):
    body = {"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]}
    return drive.files().create(body=body, fields="id", supportsAllDrives=True).execute()["id"]


def upload(path, parent_id):
    media = MediaFileUpload(path, resumable=True)
    body = {"name": os.path.basename(path), "parents": [parent_id]}
    drive.files().create(body=body, media_body=media, fields="id", supportsAllDrives=True).execute()
    print("uploaded", path)


# Satu subfolder per run supaya hasil lama tidak tertimpa
run_folder = make_folder(f"rq1_ner_{datetime.now():%Y%m%d-%H%M}", DRIVE_FOLDER_ID)

!zip -q -r /content/rq1_ner_results.zip outputs/metrics outputs/predictions
upload("/content/rq1_summary.csv", run_folder)
upload("/content/rq1_ner_results.zip", run_folder)
print(f"https://drive.google.com/drive/folders/{run_folder}")